# Handwritten Digit Recognition using FPGA Acceleration

In this lab, you will profile a software-based handwritten digit recognition pipeline to identify performance bottlenecks, and then offload the computationally expensive k-NN classification kernel to the Zynq Programmable Logic (PL) for hardware acceleration. Please refer to your assignment PDF for detailed background information, task breakdowns, and grading rubrics.

In [ ]:
# Library Declarations
import time
import random
import numpy as np
import pandas as pd 
from matplotlib import pyplot as plt
# from pynq import Overlay, allocate, MMIO

### ⚙️ **Configuration**
Update the cell below with the specific parameters for your design (e.g., section, bit-width, or AXI interface settings).

In [ ]:
RUN_SW = True
RUN_HW = False
NUM_SAMPLES = 5
NUM_FEATURES = 49
NUM_TRAIN_SAMPLES = 1500
NUM_TEST_SAMPLES = 1000

INT_TYPE = np.uint32
FLOAT_TYPE = np.float64

ADDR_OFFSET_CTRL    = 0x40000000
ADDR_OFFSET_DATA    = 0x40010000
ADDRESS_RANGE       = 0x10000
ADDR_X_train_1      = 0x1c
ADDR_X_train_2      = 0x20
ADDR_Y_train_1      = 0x28
ADDR_Y_train_2      = 0x2c
ADDR_X_test_1       = 0x10
ADDR_X_test_2       = 0x14
ADDR_Y_test_1       = 0x34
ADDR_Y_test_2       = 0x38

# ol=Overlay('./knn_optimized.bit')
# knn_data_ip=MMIO(ADDR_OFFSET_DATA,ADDRESS_RANGE)
# knn_ctrl_ip=MMIO(ADDR_OFFSET_CTRL,ADDRESS_RANGE)

####### DO NOT CHANGE
CTRL_ADDR       = 0x00  # Control register address
START_VALUE     = 1<<0  # bit 0
STOP_VALUE      = 0x0   # bit 0
DONE_VALUE      = 1<<1  # bit 1
AUTO_RESTART    = 1<<7  # bit 7

**Loading the Dataset**

The unzipped `train` and `test` data files should be included in the project demo folder. We will load the data and split it for our model.

In [ ]:
# Load the dataset
data_train = pd.read_csv("train_data.csv")
data_test = pd.read_csv("test_data.csv")

In [ ]:
train_data = data_train.values[:(NUM_TRAIN_SAMPLES+NUM_TEST_SAMPLES)]
test_data = data_test.values[:NUM_TEST_SAMPLES]

# Separate labels (column 0) and features (columns 1-784)
X, y = train_data[:,1:], train_data[:, 0]

# Split into training and testing sets (80/20 split)
split = int(X.shape[0] * 0.60)
X_train, X_valid, y_train, y_valid = X[:split], X[split:], y[:split], y[split:]

print(f"Training data shape: {X_train.shape}, Labels shape: {y_train.shape}")
print(f"Validation data shape: {X_valid.shape}, Labels shape: {y_valid.shape}")

In [ ]:
# Plotting images
a = X_train[56].reshape((28,28))
plt.figure()
plt.imshow(a)
plt.show()

**Building Block Definitions**

Here we define the core software functions for our pipeline: preprocessing the data (scaling), extracting features to reduce dimensionality (PCA), and the baseline k-NN classification algorithm for software profiling.

In [ ]:
# # 1. Pre-processing / Digitization
def preprocess_data(X_train, X_test):
    # Calculate mean and standard deviation along the feature axis (columns)
    mean = np.mean(X_train, axis=0)
    std = np.std(X_train, axis=0)
    
    # Prevent division by zero in case a feature is perfectly constant
    std[std == 0] = 1e-8
    
    # Scale the data
    X_train_scaled = (X_train - mean) / std
    X_test_scaled = (X_test - mean) / std
    
    return X_train_scaled, X_test_scaled

# # 2. Feature Extraction
def extract_features_pca(X_train, X_test, n_components):
    # 1. Center the training data (critical for PCA)
    mean = np.mean(X_train, axis=0)
    X_train_centered = X_train - mean
    
    # 2. Compute the covariance matrix
    # rowvar=False treats each column as a separate feature variable
    cov_matrix = np.cov(X_train_centered, rowvar=False)
    
    # 3. Compute eigenvalues and eigenvectors
    # np.linalg.eigh is optimized for symmetric matrices like covariance matrices
    eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)
    
    # 4. Sort eigenvectors by descending eigenvalues
    sorted_indices = np.argsort(eigenvalues)[::-1]
    sorted_eigenvectors = eigenvectors[:, sorted_indices]
    
    # 5. Extract the projection matrix (the top 'n' components)
    projection_matrix = sorted_eigenvectors[:, :n_components]
    
    # 6. Project the training data onto the new feature space
    X_train_pca = np.dot(X_train_centered, projection_matrix)
    
    # 7. Center and project the test data using the TRAIN mean and projection matrix
    X_test_centered = X_test - mean
    X_test_pca = np.dot(X_test_centered, projection_matrix)
    
    return X_train_pca, X_test_pca

# 3. Classification (Pure Python KNN for baseline profiling)
def knn_classify_python(train_features, train_labels, test_feature, k):
    distances = []
    for i in range(len(train_features)):
        dist = np.sqrt(np.sum((test_feature - train_features[i])**2))
        distances.append((train_labels[i], dist))
    
    distances.sort(key=lambda x: x[1])
    k_nearest_labels = [label for label, _ in distances[:k]]
    return max(set(k_nearest_labels), key=k_nearest_labels.count)

def digit_recognition_python(X_train, X_test, y_test):
    # TODO: Calculate timing of each step t_prep, t_pca, t_knn for profiling
    X_train_prep, X_test_prep = preprocess_data(X_train, X_test)

    X_train_pca, X_test_pca = extract_features_pca(X_train_prep, X_test_prep, n_components=NUM_FEATURES)
    
    # Combine labels (column 0) and PCA features (columns 1-15) for training data
    train_export = np.column_stack((y_train, X_train_pca))
    test_export = np.column_stack((y_test, X_test_pca))
    num_features = X_train_pca.shape[1]
    export_fmt = ['%d'] + ['%f'] * num_features
    np.savetxt("train_data_hls.dat", train_export, fmt=export_fmt, delimiter=" ")
    np.savetxt("test_data_hls.dat", test_export, fmt=export_fmt, delimiter=" ")
    print(f"Successfully exported {train_export.shape[0]} training samples and {test_export.shape[0]} testing samples for HLS simulation!")

    num_samples_to_test = len(X_test_pca) 
    correct_predictions = 0

    for i in range(len(X_test_pca)):
        pred = knn_classify_python(X_train_pca, y_train, X_test_pca[i], k=3)
        if pred == y_test[i]:
            correct_predictions += 1

    accuracy = (correct_predictions / num_samples_to_test) * 100
    return [t_prep * 1000, t_pca * 1000, t_knn * 1000, accuracy]

**Step 1: Software System & Profiling**

By measuring the execution time of each stage, we can profile their latency contributions and pinpoint the system bottleneck.

In [ ]:
if RUN_SW:
    print("--- Starting Software Execution & Profiling ---")

    [t_prep, t_pca, t_knn, accuracy] = digit_recognition_python(X_train, X_valid, y_valid)

    print(f"Software Accuracy:          {accuracy:.2f}%")
    print(f"Preprocessing Latency:      {t_prep:.2f} (ms)")
    print(f"Feature Extraction Latency: {t_pca:.2f} (ms)")
    print(f"KNN Classification Latency: {t_knn:.2f} (ms)")

    labels = ['Preprocessing', 'Feature Extraction', 'KNN Classification']
    times = [t_prep, t_pca, t_knn]
    plt.bar(labels, times, color=['blue', 'green', 'red'])
    plt.ylabel('Time (ms)')
    plt.title('Latency Contribution of Pipeline Stages')
    plt.show()

In [ ]:
if RUN_SW:
    # Perform inference on a few unlabeled images
    y_test = np.zeros(NUM_TEST_SAMPLES, dtype=np.int32)

    for i in range(0, NUM_TEST_SAMPLES):
        y_test[i] = knn_classify_python(X_train, y_train, test_data[i], k=3)

    random_indices = np.random.choice(range(0, NUM_TEST_SAMPLES), size=NUM_SAMPLES, replace=False)

    for i in range(0, NUM_SAMPLES):
        test = test_data[random_indices[i]]
        im = test.reshape((28,28))
        plt.figure()
        plt.imshow(im, cmap='gray')
        plt.show()
        print("Label:", y_test[random_indices[i]])

**Bottleneck Analysis and Hardware Offloading**

As shown by the profiling graph, the **k-NN Classification** stage dominates the execution time. Computing the Euclidean distance requires iterating over the entire training dataset for every test query.

This results in a high computational complexity of $O(N \times D)$. To achieve real-time performance, we will keep the control flow and preprocessing on the ARM processor and offload the highly parallelizable k-NN computations to a custom kernel on the FPGA side.

In [ ]:
if RUN_HW:
    # Note: Ensure your .bit and .hwh files are loaded on the PYNQ board before running.
    print("--- Starting Hardware Accelerator Execution ---")
    # TODO: Calculate timing of each step (t_prep, t_pca, t_knn) for hardware side profiling
    
    # TODO: Perform the preprocessing and feature extraction steps in software to generate the input data for the hardware kernel
    
    # take X_test_pca as input and generate y_test_out
    y_test_out = np.zeros(NUM_TEST_SAMPLES, dtype=np.int32)

    # Fill in the PYNQ overlay loading, memory allocation, and MMIO register control in 
    # this section based on your HLS block design.
    # TODO: Perform the necessary steps to adapt with your kernel data type
    
    
    # TODO: Allocate buffers for input and output data using the EXACT hardware bitwidths
    

    # TODO: Copy flattened data to the buffers
        

    # TODO: Write pointer values to specified addresses of the IP
    
    
    # TODO: Starting and stopping the IP
    

    # Calculate accuracy by comparing y_test_out with y_valid
    correct_predictions = 0
    for i in range(len(X_valid)):
        if y_test_out[i] == y_valid[i]:
            correct_predictions += 1
    accuracy = (correct_predictions / NUM_TEST_SAMPLES) * 100

    print(f"Hardware Accuracy:          {accuracy}%")   
    print(f"Preprocessing Latency:      {t_prep:.6f} (ms)")
    print(f"Feature Extraction Latency: {t_pca:.6f} (ms)")
    print(f"KNN Classification Latency: {t_knn:.6f} (ms)")

    labels = ['Preprocessing', 'Feature Extraction', 'KNN Classification']
    times = [t_prep, t_pca, t_knn]
    plt.bar(labels, times, color=['blue', 'green', 'red'])
    plt.ylabel('Time (seconds)')
    plt.title('Latency Contribution of Pipeline Stages')
    plt.show()

**Visualizing Validation Data and Predictions**

While the overall accuracy metric gives us a quantitative measure of performance, this visual check provides an intuitive understanding of how well hardware accelerator is interpreting the handwritten digits.

In [ ]:
if RUN_HW:
    random_indices = np.random.choice(range(0, NUM_TEST_SAMPLES), size=NUM_SAMPLES, replace=False)

    for i in range(0, NUM_SAMPLES):
        test = X_valid[random_indices[i]]
        im = test.reshape((28,28))
        plt.figure()
        plt.imshow(im, cmap='gray')
        plt.show()
        print("Label:", y_test_out[random_indices[i]])